# Arctic Times — Subscriber Churn Prediction

## Overview
Train a churn model on subscriber behavior with Snowpark + scikit-learn, then deploy it as a SQL-callable Python UDTF — all inside Snowflake.

## What You'll Learn
- Feature engineering on Snowpark DataFrames (no data movement)
- Training a GradientBoosting churn model in the notebook
- Deploying the model as `ARCTIC_TIMES.ML.PREDICT_CHURN` (a SQL UDTF)
- Scoring subscribers directly from SQL

## Prerequisites
- `ARCTIC_TIMES.RAW.SUBSCRIBERS` populated (see `scripts/deploy/`)
- Warehouse `COMPUTE_WH`; packages: `snowflake-snowpark-python`, `scikit-learn`

## Estimated Time: ~5 minutes

In [ ]:
# === ALL IMPORTS GO HERE ===
import logging
import time
import os
import numpy as np
import pandas as pd
import streamlit as st
import joblib
import sklearn
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import snowflake.snowpark.functions as F
from snowflake.snowpark.context import get_active_session

# Active session (provided by Snowflake Notebooks)
session = get_active_session()
session.use_database("ARCTIC_TIMES")
session.use_schema("RAW")

# Observability
logging.getLogger().setLevel(logging.INFO)
logger = logging.getLogger("churn_demo")
logger.info(f"Connected to {session.get_current_database()}.{session.get_current_schema()} on {session.get_current_warehouse()}")

## 1. Feature Engineering
Build training features from subscriber behavior — computed inside Snowflake, then pulled to pandas for training.

In [ ]:
features_df = session.table("ARCTIC_TIMES.RAW.SUBSCRIBERS").select(
    F.col("USER_ID"),
    F.datediff("day", F.col("LAST_LOGIN"), F.current_date()).alias("DAYS_SINCE_LOGIN"),
    F.col("ARTICLES_READ_30D"),
    F.datediff("month", F.col("START_DATE"), F.current_date()).alias("TENURE_MONTHS"),
    F.col("AVG_SESSION_SEC"),
    F.col("PAYWALL_BOUNCES_30D"),
    F.col("SUBSCRIPTION_TYPE"),
    F.col("SEGMENT"),
    F.col("CHURN_FLAG").cast("int").alias("CHURNED"),
)

pdf = features_df.to_pandas()
logger.info(f"Training dataset: {len(pdf):,} subscribers | churn rate {pdf['CHURNED'].mean():.1%}")
st.dataframe(pdf.head())

## 2. Model Training
A GradientBoosting classifier on the engineered features. Categorical subscription type is label-encoded.

In [ ]:
start = time.time()

pdf["SUB_TYPE_ENC"] = pdf["SUBSCRIPTION_TYPE"].map(
    {"student": 0, "digital_only": 1, "standard": 2, "premium": 3}
).fillna(2)

feature_cols = [
    "DAYS_SINCE_LOGIN", "ARTICLES_READ_30D", "TENURE_MONTHS",
    "AVG_SESSION_SEC", "PAYWALL_BOUNCES_30D", "SUB_TYPE_ENC",
]
X = pdf[feature_cols].fillna(0)
y = pdf["CHURNED"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = GradientBoostingClassifier(
    n_estimators=100, max_depth=5, learning_rate=0.1, subsample=0.8, random_state=42
)
model.fit(X_train, y_train)

y_proba = model.predict_proba(X_test)[:, 1]
logger.info(f"Trained in {time.time() - start:.1f}s | ROC AUC {roc_auc_score(y_test, y_proba):.3f}")
st.text(classification_report(y_test, model.predict(X_test), target_names=["Active", "Churned"]))

# Pin the UDTF's scikit-learn to this runtime's version so the
# serialized model unpickles cleanly server-side.
sklearn_ver = sklearn.__version__
logger.info(f"scikit-learn {sklearn_ver} (used to pin the UDTF)")

## 3. What Drives Churn?
Feature importances from the trained model.

In [ ]:
importance = pd.DataFrame(
    {"feature": feature_cols, "importance": model.feature_importances_}
).sort_values("importance", ascending=False)

st.subheader("Feature importance")
st.bar_chart(importance, x="feature", y="importance")
logger.info(f"Top churn driver: {importance.iloc[0]['feature']}")

## 4. Deploy as a SQL UDTF
Serialize the model to a stage and register it as `ARCTIC_TIMES.ML.PREDICT_CHURN`. Any analyst can then call it from SQL — no Python required.

In [ ]:
model_path = "/tmp/churn_model.pkl"
joblib.dump(model, model_path)
logger.info(f"Model serialized: {os.path.getsize(model_path) / 1024:.1f} KB")

session.file.put(model_path, "@ARCTIC_TIMES.ML.MODELS", auto_compress=False, overwrite=True)
logger.info("Uploaded to @ARCTIC_TIMES.ML.MODELS/churn_model.pkl")

In [ ]:
CREATE OR REPLACE FUNCTION ARCTIC_TIMES.ML.PREDICT_CHURN(
    days_since_last_login INT, articles_read_30d INT,
    subscription_months INT, avg_session_sec FLOAT, paywall_bounces_30d INT
)
RETURNS TABLE (churn_probability FLOAT, risk_segment VARCHAR)
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('scikit-learn=={{sklearn_ver}}', 'pandas', 'numpy', 'joblib', 'cachetools')
IMPORTS = ('@ARCTIC_TIMES.ML.MODELS/churn_model.pkl')
HANDLER = 'ChurnPredictor'
AS $$
import numpy as np, joblib, sys, os

class ChurnPredictor:
    def __init__(self):
        import_dir = sys._xoptions.get("snowflake_import_directory", "/tmp")
        self.model = joblib.load(os.path.join(import_dir, "churn_model.pkl"))

    def process(self, days_since_last_login, articles_read_30d,
                subscription_months, avg_session_sec, paywall_bounces_30d):
        features = np.array([[
            days_since_last_login or 0, articles_read_30d or 0,
            subscription_months or 0, avg_session_sec or 0.0,
            paywall_bounces_30d or 0, 2
        ]])
        prob = float(self.model.predict_proba(features)[0][1])
        segment = 'HIGH' if prob > 0.7 else 'MEDIUM' if prob > 0.4 else 'LOW'
        yield (round(prob, 4), segment)
$$;

## 5. Score Subscribers from SQL
Find high-value subscribers at churn risk — a pure SQL call to the UDTF.

In [ ]:
SELECT s.user_id, s.subscription_type, s.ltv_estimated_eur,
       c.churn_probability, c.risk_segment
FROM ARCTIC_TIMES.RAW.SUBSCRIBERS s,
     TABLE(ARCTIC_TIMES.ML.PREDICT_CHURN(
         DATEDIFF('day', s.last_login, CURRENT_DATE()),
         s.articles_read_30d,
         DATEDIFF('month', s.start_date, CURRENT_DATE()),
         s.avg_session_sec::FLOAT,
         s.paywall_bounces_30d
     )) c
WHERE c.risk_segment = 'HIGH'
ORDER BY s.ltv_estimated_eur DESC
LIMIT 10;

In [ ]:
# Reference the SQL cell result (Snowpark DataFrame) and display it
risk = top_risk_subscribers.to_pandas()
st.subheader("Top 10 high-value subscribers at risk")
st.dataframe(risk)
logger.info(f"{len(risk)} high-risk high-value subscribers flagged for re-engagement")

## Cleanup

Set `RUN_CLEANUP = True` in the next cell to remove the objects this notebook created (model file + UDTF). It is `False` by default so the UDTF stays deployed.

In [ ]:
# Set to True to tear down the demo objects (model file + UDTF).
# Left False so a headless run (snow notebook execute) keeps the UDTF deployed.
RUN_CLEANUP = False

cleanup_stmts = [
    "DROP FUNCTION IF EXISTS ARCTIC_TIMES.ML.PREDICT_CHURN(INT, INT, INT, FLOAT, INT)",
    "REMOVE @ARCTIC_TIMES.ML.MODELS/churn_model.pkl",
]
if RUN_CLEANUP:
    for stmt in cleanup_stmts:
        try:
            session.sql(stmt).collect()
            logger.info(f"Executed: {stmt}")
        except Exception as e:
            logger.warning(f"Cleanup warning: {e}")
    logger.info("Cleanup complete.")
else:
    logger.info("RUN_CLEANUP=False — UDTF and model left in place. Set True to tear down.")